In [1]:
import os

PROJECT = "/content/warp_reduction"
SRC = os.path.join(PROJECT, "src")
os.makedirs(SRC, exist_ok=True)

In [2]:
%%writefile /content/warp_reduction/src/baseline_shared.cu
#include <iostream>
#include <vector>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " \
                  << cudaGetErrorString(err) \
                  << std::endl; \
        return 1; \
    } \
} while (0)

__global__ void reduceShared(
    const int* input,
    int* output,
    int n
) {
    __shared__ int partial[256];

    int tid = threadIdx.x;
    int i = tid;

    // Load one value per thread; use zero for out-of-range.
    partial[tid] = (i < n) ? input[i] : 0;
    __syncthreads();

    // Tree reduction in shared memory.
    for (int stride = blockDim.x / 2;
         stride > 0;
         stride /= 2) {

        if (tid < stride) {
            partial[tid] += partial[tid + stride];
        }

        __syncthreads();
    }

    // Thread 0 holds the final sum.
    if (tid == 0) {
        output[0] = partial[0];
    }
}

int main() {
    const int n = 8;
    std::vector<int> h_input = {
        10, 20, 30, 40, 50, 60, 70, 80
    };

    int h_output = 0;
    int *d_input = nullptr;
    int *d_output = nullptr;

    CHECK(cudaMalloc(&d_input, n * sizeof(int)));
    CHECK(cudaMalloc(&d_output, sizeof(int)));

    CHECK(cudaMemcpy(
        d_input, h_input.data(), n * sizeof(int),
        cudaMemcpyHostToDevice
    ));

    int threads = 256;

    reduceShared<<<1, threads>>>(d_input, d_output, n);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(
        &h_output, d_output, sizeof(int),
        cudaMemcpyDeviceToHost
    ));

    std::cout << "Shared-memory sum: "
              << h_output << std::endl;

    CHECK(cudaFree(d_input));
    CHECK(cudaFree(d_output));

    return 0;
}

Writing /content/warp_reduction/src/baseline_shared.cu


In [3]:
!nvcc /content/warp_reduction/src/baseline_shared.cu \
    -o /content/warp_reduction/src/baseline_shared

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!/content/warp_reduction/src/baseline_shared

Shared-memory sum: 360
